preprocessing

In [1]:
import os
import torch
from torch import nn, optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import random

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# CONFIG
DATA_PATH = './kfold'
TARGET_DOMAIN = 'art_painting'
BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 20
NUM_WORKERS = 4

all_domains = ['art_painting', 'cartoon', 'photo', 'sketch']
train_domains = [d for d in all_domains if d != TARGET_DOMAIN]

# DATASET AND TRANSFORMS
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load datasets for each domain
domain_datasets = {
    domain: datasets.ImageFolder(os.path.join(DATA_PATH, domain), transform=transform)
    for domain in all_domains
}

# Split source domains into train/val
def split_dataset(dataset, val_ratio=0.2):
    indices = list(range(len(dataset)))
    train_idx, val_idx = train_test_split(indices, test_size=val_ratio, random_state=42)
    return Subset(dataset, train_idx), Subset(dataset, val_idx)

train_loaders = {}
val_loaders = {}

for domain in train_domains:
    train_subset, val_subset = split_dataset(domain_datasets[domain])
    train_loaders[domain] = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loaders[domain] = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Target test loader (art domain)
test_loader = DataLoader(domain_datasets[TARGET_DOMAIN], batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# MODEL: Simple pretrained ResNet
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, len(domain_datasets[TARGET_DOMAIN].classes))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# Helper to combine domain batches
def get_domainbed_batch(train_loaders_iter):
    images, labels = [], []
    for loader in train_loaders_iter:
        try:
            x, y = next(loader)
        except StopIteration:
            continue
        images.append(x)
        labels.append(y)
    return torch.cat(images).to(DEVICE), torch.cat(labels).to(DEVICE)

# Training Loop
def train():
    best_val_acc = 0
    for epoch in range(EPOCHS):
        model.train()
        train_iters = [iter(train_loaders[domain]) for domain in train_domains]

        while True:
            if all([len(loader) == 0 for loader in train_iters]):
                break
            try:
                x_batch, y_batch = get_domainbed_batch(train_iters)
            except:
                break
            optimizer.zero_grad()
            out = model(x_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()

        # Validation after each epoch
        val_acc = validate(val_loaders)
        print(f"Epoch {epoch+1}/{EPOCHS} | Val Acc: {val_acc:.2f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")

# Validation Accuracy
def validate(val_loaders):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for loader in val_loaders.values():
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds = model(x).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
    return correct / total * 100

# Test Accuracy
def test():
    model.load_state_dict(torch.load("best_model.pth"))
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    print(f"Test Accuracy on {TARGET_DOMAIN}: {correct / total * 100:.2f}%")

# Run
train()
test()


C:\Users\Lenovo\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Lenovo\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/20 | Val Acc: 64.82
Epoch 2/20 | Val Acc: 74.45
Epoch 3/20 | Val Acc: 76.46
Epoch 4/20 | Val Acc: 76.34
Epoch 5/20 | Val Acc: 77.72
Epoch 6/20 | Val Acc: 83.26
Epoch 7/20 | Val Acc: 83.13
Epoch 8/20 | Val Acc: 78.98
Epoch 9/20 | Val Acc: 81.75
Epoch 10/20 | Val Acc: 81.69
Epoch 11/20 | Val Acc: 79.42
Epoch 12/20 | Val Acc: 76.65
Epoch 13/20 | Val Acc: 80.74
Epoch 14/20 | Val Acc: 71.49
Epoch 15/20 | Val Acc: 80.93
Epoch 16/20 | Val Acc: 85.15
Epoch 17/20 | Val Acc: 81.25
Epoch 18/20 | Val Acc: 85.40
Epoch 19/20 | Val Acc: 87.04
Epoch 20/20 | Val Acc: 87.54
Test Accuracy on art_painting: 37.99%


In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
print(torch.cuda.device_count())
print(torch.cuda.current_device())

1
0


In [5]:
DEVICE = torch.device("cuda")
DEVICE

device(type='cuda')

In [1]:
import os
import torch
from torch import nn, optim
from torchvision import datasets, models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import random
from collections import OrderedDict

ERM

In [ ]:
import os
import torch
from torch import nn, optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import random

# ======= CONFIG =======
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_PATH = './kfold'  
TARGET_DOMAIN = 'art_painting'
BATCH_SIZE = 32  # Per domain
TOTAL_ITERS = 5000
LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEED = 42
VAL_SPLIT = 0.2

# ======= SEEDING =======
torch.manual_seed(SEED)
random.seed(SEED)

# ======= DOMAINS =======
all_domains = ['art_painting', 'cartoon', 'photo', 'sketch']
train_domains = [d for d in all_domains if d != TARGET_DOMAIN]

# ======= TRANSFORMS =======
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ======= LOAD DATASETS =======
domain_datasets = {
    domain: datasets.ImageFolder(os.path.join(DATA_PATH, domain), transform=transform)
    for domain in all_domains
}

def split_dataset(dataset, val_ratio=VAL_SPLIT):
    indices = list(range(len(dataset)))
    train_idx, val_idx = train_test_split(indices, test_size=val_ratio, random_state=SEED)
    return Subset(dataset, train_idx), Subset(dataset, val_idx)

train_loaders = {}
val_loaders = {}

for domain in train_domains:
    train_subset, val_subset = split_dataset(domain_datasets[domain])
    train_loaders[domain] = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                                       num_workers=NUM_WORKERS, drop_last=True)
    val_loaders[domain] = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                                     num_workers=NUM_WORKERS)

# Target test loader (for art_painting)
test_loader = DataLoader(domain_datasets[TARGET_DOMAIN], batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS)

# ======= MODEL =======
model = models.resnet18(pretrained=True)
num_classes = len(domain_datasets[TARGET_DOMAIN].classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)

# ======= OPTIMIZER & LOSS =======
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

# ======= TRAINING UTIL =======
def get_domainbed_batch(train_iters):
    images, labels = [], []
    for it in train_iters:
        try:
            x, y = next(it)
            images.append(x)
            labels.append(y)
        except StopIteration:
            continue
    return torch.cat(images).to(DEVICE), torch.cat(labels).to(DEVICE)

def validate():
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for loader in val_loaders.values():
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds = model(x).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
    return correct / total * 100

# ======= TRAIN LOOP =======
def train():
    best_val_acc = 0
    train_iters = [iter(train_loaders[domain]) for domain in train_domains]

    for step in range(1, TOTAL_ITERS + 1):
        if any([len(it.dataset) == 0 for it in train_loaders.values()]):
            break

        batch = get_domainbed_batch(train_iters)
        if len(batch[0]) < BATCH_SIZE * len(train_domains):
            train_iters = [iter(train_loaders[d]) for d in train_domains]
            continue

        model.train()
        x, y = batch
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        if step % 500 == 0:
            val_acc = validate()
            print(f"[{step}/{TOTAL_ITERS}] Validation Acc: {val_acc:.2f}%")
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), 'best_resnet18_model.pth')

# ======= TEST LOOP =======
def test():
    model.load_state_dict(torch.load('best_resnet18_model.pth'))
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    print(f" Test Accuracy on {TARGET_DOMAIN}: {correct / total * 100:.2f}%")

# ======= RUN =======
train()
test()


[1/5000] Validation Acc: 23.35%
[501/5000] Validation Acc: 94.97%
[1001/5000] Validation Acc: 95.78%
[1501/5000] Validation Acc: 95.66%
[2001/5000] Validation Acc: 95.91%
[2501/5000] Validation Acc: 95.41%
[3001/5000] Validation Acc: 95.85%
[3501/5000] Validation Acc: 95.78%
[4001/5000] Validation Acc: 95.85%
[4501/5000] Validation Acc: 95.78%
🎯 Test Accuracy on art_painting: 78.22%


Freezing

In [6]:
# ======= MODEL (Freeze BN layers) =======
model = models.resnet18(pretrained=True)
num_classes = len(domain_datasets[TARGET_DOMAIN].classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Freeze BatchNorm layers
for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.eval()              # turn off running stats
        for param in module.parameters():
            param.requires_grad = False

model = model.to(DEVICE)


In [7]:
train()
test()


[1/5000] Validation Acc: 16.74%
[501/5000] Validation Acc: 18.00%
[1001/5000] Validation Acc: 18.31%
[1501/5000] Validation Acc: 18.06%
[2001/5000] Validation Acc: 18.00%
[2501/5000] Validation Acc: 18.00%
[3001/5000] Validation Acc: 18.19%
[3501/5000] Validation Acc: 18.19%
[4001/5000] Validation Acc: 18.00%
[4501/5000] Validation Acc: 18.38%
🎯 Test Accuracy on art_painting: 21.58%


ADAM optimizer

In [ ]:
import os
import torch
from torch import nn, optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import random

# ======= CONFIG =======
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_PATH = './kfold'  
TARGET_DOMAIN = 'art_painting'
BATCH_SIZE = 32  # Per domain
TOTAL_ITERS = 5000
LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEED = 42
VAL_SPLIT = 0.2

# ======= SEEDING =======
torch.manual_seed(SEED)
random.seed(SEED)

# ======= DOMAINS =======
all_domains = ['art_painting', 'cartoon', 'photo', 'sketch']
train_domains = [d for d in all_domains if d != TARGET_DOMAIN]

# ======= TRANSFORMS =======
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ======= LOAD DATASETS =======
domain_datasets = {
    domain: datasets.ImageFolder(os.path.join(DATA_PATH, domain), transform=transform)
    for domain in all_domains
}

def split_dataset(dataset, val_ratio=VAL_SPLIT):
    indices = list(range(len(dataset)))
    train_idx, val_idx = train_test_split(indices, test_size=val_ratio, random_state=SEED)
    return Subset(dataset, train_idx), Subset(dataset, val_idx)

train_loaders = {}
val_loaders = {}

for domain in train_domains:
    train_subset, val_subset = split_dataset(domain_datasets[domain])
    train_loaders[domain] = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                                       num_workers=NUM_WORKERS, drop_last=True)
    val_loaders[domain] = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                                     num_workers=NUM_WORKERS)

# Target test loader (for art_painting)
test_loader = DataLoader(domain_datasets[TARGET_DOMAIN], batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS)

# ======= MODEL =======
model = models.resnet18(pretrained=True)
num_classes = len(domain_datasets[TARGET_DOMAIN].classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)

# ======= OPTIMIZER & LOSS =======
# NEW:
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

# ======= TRAINING UTIL =======
def get_domainbed_batch(train_iters):
    images, labels = [], []
    for it in train_iters:
        try:
            x, y = next(it)
            images.append(x)
            labels.append(y)
        except StopIteration:
            continue
    return torch.cat(images).to(DEVICE), torch.cat(labels).to(DEVICE)

def validate():
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for loader in val_loaders.values():
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds = model(x).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
    return correct / total * 100

# ======= TRAIN LOOP =======
def train():
    best_val_acc = 0
    train_iters = [iter(train_loaders[domain]) for domain in train_domains]

    for step in range(1, TOTAL_ITERS + 1):
        if any([len(it.dataset) == 0 for it in train_loaders.values()]):
            break

        batch = get_domainbed_batch(train_iters)
        if len(batch[0]) < BATCH_SIZE * len(train_domains):
            train_iters = [iter(train_loaders[d]) for d in train_domains]
            continue

        model.train()
        x, y = batch
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        if step % 250 == 0:
            val_acc = validate()
            print(f"[{step}/{TOTAL_ITERS}] Val Acc: {val_acc:.2f}%")
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), 'best_resnet18_model.pth')

# ======= TEST LOOP =======
def test():
    model.load_state_dict(torch.load('best_resnet18_model.pth'))
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    print(f" Test Accuracy on {TARGET_DOMAIN}: {correct / total * 100:.2f}%")

# ======= RUN =======
train()
test()


[250/5000] Val Acc: 94.71%
[500/5000] Val Acc: 94.08%
[750/5000] Val Acc: 95.34%
[1000/5000] Val Acc: 95.15%
[1250/5000] Val Acc: 95.47%
[1500/5000] Val Acc: 95.53%
[1750/5000] Val Acc: 95.47%
[2000/5000] Val Acc: 95.53%
[2250/5000] Val Acc: 95.59%
[2500/5000] Val Acc: 95.47%
[2750/5000] Val Acc: 95.41%
[3000/5000] Val Acc: 95.47%
[3250/5000] Val Acc: 95.47%
[3500/5000] Val Acc: 95.59%
[3750/5000] Val Acc: 95.59%
[4000/5000] Val Acc: 95.59%
[4250/5000] Val Acc: 95.53%
[4500/5000] Val Acc: 95.47%
[4750/5000] Val Acc: 95.85%
[5000/5000] Val Acc: 95.97%
🎯 Test Accuracy on art_painting: 78.27%


FAD

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader
from itertools import cycle

# ======================== CONFIG ========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
NUM_CLASSES = 7  # Assuming 7 classes in PACS dataset
LR = 1e-3
WEIGHT_DECAY = 1e-4
DAMPING_FACTOR = 0.1
NUM_ITERS = 5000
PRINT_EVERY = 500
DATA_ROOT = './kfold'  # Root path to PACS dataset
SOURCE_DOMAINS = ['photo', 'cartoon', 'sketch']
TARGET_DOMAIN = 'art_painting'

# ======================== TRANSFORM ========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ======================== DATA LOADERS ========================
train_loaders = {}
train_iters = []
for domain in SOURCE_DOMAINS:
    path = os.path.join(DATA_ROOT, domain)
    dataset = datasets.ImageFolder(path, transform=transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    train_loaders[domain] = loader
    train_iters.append(cycle(loader))

# Target domain loader for evaluation
target_path = os.path.join(DATA_ROOT, TARGET_DOMAIN)
target_dataset = datasets.ImageFolder(target_path, transform=transform)
target_loader = DataLoader(target_dataset, batch_size=64, shuffle=False, num_workers=4)

# ======================== MODEL ========================
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

# ======================== FAD OPTIMIZER ========================
class FADOptimizer(optim.Optimizer):
    def __init__(self, params, lr=0.001, weight_decay=0.0001, damping_factor=0.1):
        defaults = dict(lr=lr, weight_decay=weight_decay)
        super(FADOptimizer, self).__init__(params, defaults)
        self.base_optimizer = optim.SGD(self.param_groups, lr=lr, weight_decay=weight_decay)
        self.damping_factor = damping_factor

    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            for param in group['params']:
                if param.grad is not None:
                    param.grad.data.mul_(self.damping_factor)

        self.base_optimizer.step()
        return loss

optimizer = FADOptimizer(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, damping_factor=DAMPING_FACTOR)
criterion = nn.CrossEntropyLoss()

# ======================== TRAIN ========================
model.train()
iter_count = 0
while iter_count < NUM_ITERS:
    all_images, all_labels = [], []
    for domain_iter in train_iters:
        images, labels = next(domain_iter)
        all_images.append(images)
        all_labels.append(labels)
    images = torch.cat(all_images).to(DEVICE)
    labels = torch.cat(all_labels).to(DEVICE)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    iter_count += 1
    if iter_count % PRINT_EVERY == 0:
        _, preds = torch.max(outputs, 1)
        acc = (preds == labels).float().mean()
        print(f"[Iter {iter_count}] Loss: {loss.item():.4f} | Acc: {acc.item() * 100:.2f}%")

# ======================== EVAL ON TARGET DOMAIN ========================
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in target_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f" Final Accuracy on Target Domain ({TARGET_DOMAIN}): {100 * correct / total:.2f}%")

C:\Users\Lenovo\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Lenovo\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[Iter 500] Loss: 1.7112 | Acc: 31.25%
[Iter 1000] Loss: 1.4209 | Acc: 52.08%
[Iter 1500] Loss: 1.2062 | Acc: 68.75%
[Iter 2000] Loss: 1.0837 | Acc: 69.79%
[Iter 2500] Loss: 0.9946 | Acc: 78.12%
[Iter 3000] Loss: 0.8753 | Acc: 78.12%
[Iter 3500] Loss: 0.7401 | Acc: 86.46%
[Iter 4000] Loss: 0.7128 | Acc: 85.42%
[Iter 4500] Loss: 0.6690 | Acc: 84.38%
[Iter 5000] Loss: 0.6215 | Acc: 90.62%
🎯 Final Accuracy on Target Domain (art_painting): 60.55%


In [6]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, transforms, models
from torch.func import functional_call  # Using torch.func.functional_call (PyTorch 2.0+)
from torch.cuda.amp import autocast  # For mixed precision

# -------------------------------
# Hyperparameters and settings
# -------------------------------
DEVICE = torch.device("cuda")
BATCH_SIZE = 32
NUM_CLASSES = 7       # PACS has 7 classes
NUM_ITERS = 5000
PRINT_EVERY = 500
DATA_ROOT = './kfold'
SOURCE_DOMAINS = ['photo', 'cartoon', 'sketch']
TARGET_DOMAIN = 'art_painting'

# FAD hyperparameters from paper
ETA_0 = 5e-5         # Initial learning rate
RHO_0 = 0.05         # Initial perturbation radius
ALPHA = 0.5          # Trade-off between flatness types
BETA = 1.0           # Regularization weight
XI = 1e-12           # Stability constant

# -------------------------------
# Helper functions
# -------------------------------
def get_named_parameters(model):
    """Returns a dictionary mapping parameter names to tensors."""
    return {name: param for name, param in model.named_parameters()}

def dict_norm(grad_dict):
    """Computes the global L2 norm of a dictionary of tensors."""
    squared_sum = 0.0
    for g in grad_dict.values():
        squared_sum += torch.sum(g.pow(2))
    return torch.sqrt(squared_sum)

def dict_sub(dict1, dict2):
    """Element-wise subtraction of two dictionaries."""
    return {k: dict1[k] - dict2[k] for k in dict1}

def compute_gradients(model, params, inputs, targets, criterion, create_graph=False):
    """
    Computes the loss and gradients with respect to params using functional_call.
    Forward pass is wrapped in autocast for mixed precision.
    Returns the loss and a dictionary of gradients.
    """
    with autocast():
        outputs = functional_call(model, params, (inputs,))
        loss = criterion(outputs, targets)
    grads_tuple = torch.autograd.grad(loss, tuple(params.values()), create_graph=create_graph)
    grads = {name: g for (name, _), g in zip(params.items(), grads_tuple)}
    return loss, grads

def update_model_params(model, new_params):
    """Updates model parameters in-place using new_params dictionary."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            param.copy_(new_params[name])

# -------------------------------
# Dataset preparation (ImageFolder)
# -------------------------------
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

datasets_list = []
for domain in SOURCE_DOMAINS:
    domain_path = os.path.join(DATA_ROOT, domain)
    if os.path.isdir(domain_path):
        ds = datasets.ImageFolder(root=domain_path, transform=transform_train)
        datasets_list.append(ds)
    else:
        print(f"Warning: Domain folder {domain_path} not found.")

if not datasets_list:
    raise ValueError("No valid domain folders found in DATA_ROOT.")

train_dataset = ConcatDataset(datasets_list)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

# -------------------------------
# Model and criterion
# -------------------------------
# Using the recommended weights parameter to avoid deprecation warnings.
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

# Optionally compile the model (PyTorch 2.0+)
try:
    model = torch.compile(model)
    print("Model compiled successfully.")
except Exception as e:
    print("Model compilation not supported or failed:", e)

criterion = nn.CrossEntropyLoss()

# -------------------------------
# Training loop with FAD update and cleanup
# -------------------------------
print("Starting training with FAD...")
model.train()
data_iterator = iter(train_loader)
iteration = 0

while iteration < NUM_ITERS:
    try:
        inputs, targets = next(data_iterator)
    except StopIteration:
        data_iterator = iter(train_loader)
        inputs, targets = next(data_iterator)
    
    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
    current_params = get_named_parameters(model)
    
    # ----- Step 1: Compute grad0 -----
    loss0, grad0 = compute_gradients(model, current_params, inputs, targets, criterion, create_graph=False)
    norm_grad0 = dict_norm(grad0)
    
    # ----- Step 2: Compute theta_tilde_1 and grad1 -----
    params1 = {name: param + (RHO_0 / (norm_grad0 + XI)) * grad0[name] 
               for name, param in current_params.items()}
    loss1, grad1 = compute_gradients(model, params1, inputs, targets, criterion, create_graph=False)
    
    # h0 = grad1 - grad0
    h0 = dict_sub(grad1, grad0)
    norm_h0 = dict_norm(h0)
    
    # ----- Step 3: Compute theta_tilde_2 and grad2 -----
    params2 = {name: param + (RHO_0 / (norm_h0 + XI)) * h0[name]
               for name, param in current_params.items()}
    loss2, grad2 = compute_gradients(model, params2, inputs, targets, criterion, create_graph=False)
    
    # ----- Step 4: Compute theta_tilde_3 and grad3 -----
    norm_grad2 = dict_norm(grad2)
    params3 = {name: params2[name] + (RHO_0 / (norm_grad2 + XI)) * grad2[name]
               for name in params2}
    loss3, grad3 = compute_gradients(model, params3, inputs, targets, criterion, create_graph=False)
    
    # h1 = grad3 - grad2
    h1 = dict_sub(grad3, grad2)
    
    # ----- Step 5: Combine gradients -----
    update_grad = {name: grad0[name] + BETA * (ALPHA * h0[name] + (1 - ALPHA) * h1[name])
                   for name in grad0}
    new_params = {name: current_params[name] - ETA_0 * update_grad[name]
                  for name in current_params}
    
    update_model_params(model, new_params)
    
    # Explicitly clean up intermediate variables
    del loss0, loss1, loss2, loss3
    del grad0, grad1, grad2, grad3
    del h0, h1
    del current_params, params1, params2, params3, update_grad
    
    # Periodically clear CUDA cache
    if (iteration + 1) % 100 == 0:
        torch.cuda.empty_cache()
    
    if (iteration + 1) % PRINT_EVERY == 0:
        with torch.no_grad():
            outputs = model(inputs)
            curr_loss = criterion(outputs, targets)
            preds = outputs.argmax(dim=1)
            correct = (preds == targets).sum().item()
            acc = correct / targets.size(0) * 100.0
        print(f"Iteration {iteration+1}/{NUM_ITERS}, Loss: {curr_loss.item():.4f}, Accuracy: {acc:.2f}%")
    
    iteration += 1

print("Training completed.")


Model compiled successfully.
Starting training with FAD...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6896\2997879792.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Iteration 500/5000, Loss: 1.7224, Accuracy: 37.50%
Iteration 1000/5000, Loss: 1.8524, Accuracy: 21.88%
Iteration 1500/5000, Loss: 1.5740, Accuracy: 50.00%
Iteration 2000/5000, Loss: 1.5112, Accuracy: 56.25%
Iteration 2500/5000, Loss: 1.4686, Accuracy: 56.25%
Iteration 3000/5000, Loss: 1.3049, Accuracy: 62.50%
Iteration 3500/5000, Loss: 1.1306, Accuracy: 65.62%
Iteration 4000/5000, Loss: 1.1399, Accuracy: 65.62%
Iteration 4500/5000, Loss: 1.2163, Accuracy: 65.62%
Iteration 5000/5000, Loss: 0.9868, Accuracy: 81.25%
Training completed.


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import models, transforms, datasets

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Model with Feature Extraction
# -------------------------------
class ResNet18WithFeatures(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Load a pretrained ResNet18 using recommended weights
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # Remove the final fully-connected layer to obtain features
        self.features = nn.Sequential(*list(backbone.children())[:-1])  # Output shape: (B, 512, 1, 1)
        # New classifier head for target number of classes
        self.classifier = nn.Linear(backbone.fc.in_features, num_classes)
        
    def forward(self, x):
        # Extract features
        f = self.features(x)       # (B, 512, 1, 1)
        f = torch.flatten(f, 1)    # (B, 512)
        logits = self.classifier(f)
        return logits, f

# -------------------------------
# MIRO Regularization Module
# -------------------------------
class MIRORegularizer(nn.Module):
    def __init__(self, num_classes, feature_dim):
        """
        For each class, learnable parameters for the "oracle" embedding:
         - mu: mean vector (initialized to zeros by default)
         - logvar: log of the diagonal covariance
        """
        super().__init__()
        self.mu = nn.Parameter(torch.zeros(num_classes, feature_dim))
        self.logvar = nn.Parameter(torch.zeros(num_classes, feature_dim))
        
    def forward(self, features, labels):
        """
        For each sample compute the KL divergence between 
        N(features, I) and N(mu[label], diag(exp(logvar[label]))).
        
        KL divergence for two Gaussians N(m0, I) and N(m1, V1):
          KL = 0.5 * [ - log|V1| - d + tr(V1⁻¹) + (m1 - m0)ᵀ V1⁻¹ (m1 - m0) ]
        """
        mu_target = self.mu[labels]              # (B, feature_dim)
        logvar_target = self.logvar[labels]        # (B, feature_dim)
        var_target = torch.exp(logvar_target)      # (B, feature_dim)
        d = features.size(1)
        kl_per_sample = 0.5 * (
            - torch.sum(logvar_target, dim=1) - d +
            torch.sum(1.0 / var_target, dim=1) +
            torch.sum((mu_target - features) ** 2 / var_target, dim=1)
        )
        return kl_per_sample.mean()

# -------------------------------
# Hyperparameters and Data Setup
# -------------------------------
NUM_CLASSES = 7          # for example, PACS has 7 classes
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 20              # You can adjust the number of epochs
LAMBDA_MIRO = 0.1        # Weight for the MIRO regularization term

# Set up transforms and load the dataset from the existing "kfold" folder.
# We assume that "kfold" is organized in ImageFolder format (one subfolder per class)
DATA_ROOT = './kfold'
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
# Use the entire kfold folder for training/evaluation.
dataset = datasets.ImageFolder(DATA_ROOT, transform=transform_train)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

# -------------------------------
# Initialize Model, MIRO Regularizer, Optimizer, and Loss
# -------------------------------
model = ResNet18WithFeatures(num_classes=NUM_CLASSES).to(DEVICE)
miro_reg = MIRORegularizer(num_classes=NUM_CLASSES, feature_dim=512).to(DEVICE)

# Adam optimizer updates both model and MIRO parameters.
optimizer = optim.Adam(list(model.parameters()) + list(miro_reg.parameters()), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# -------------------------------
# Function to Evaluate Model Accuracy
# -------------------------------
def evaluate(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits, _ = model(inputs)
            preds = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    model.train()
    return 100.0 * correct / total

# -------------------------------
# Training Loop
# -------------------------------
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        # Forward pass: compute logits and features
        logits, features = model(inputs)
        loss_cls = criterion(logits, labels)
        
        # MIRO regularization loss
        loss_miro = miro_reg(features, labels)
        
        # Total loss: classification loss plus weighted MIRO loss
        loss = loss_cls + LAMBDA_MIRO * loss_miro
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader)
    acc = evaluate(model, train_loader, DEVICE)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss:.4f}, Accuracy: {acc:.2f}%")
    
final_acc = evaluate(model, train_loader, DEVICE)
print(f"Final Accuracy on kfold: {final_acc:.2f}%")


Epoch 1/20, Loss: 6.0148, Accuracy: 93.92%
Epoch 2/20, Loss: 0.4569, Accuracy: 97.77%
Epoch 3/20, Loss: -1.6432, Accuracy: 98.61%
Epoch 4/20, Loss: -3.5052, Accuracy: 99.53%
Epoch 5/20, Loss: -5.2355, Accuracy: 99.82%
Epoch 6/20, Loss: -6.8585, Accuracy: 99.81%
Epoch 7/20, Loss: -8.4050, Accuracy: 99.47%
Epoch 8/20, Loss: -9.8853, Accuracy: 99.96%
Epoch 9/20, Loss: -11.3158, Accuracy: 99.43%
Epoch 10/20, Loss: -12.7023, Accuracy: 99.95%
Epoch 11/20, Loss: -14.0699, Accuracy: 99.98%
Epoch 12/20, Loss: -15.3935, Accuracy: 99.94%
Epoch 13/20, Loss: -16.6736, Accuracy: 100.00%
Epoch 14/20, Loss: -17.9490, Accuracy: 99.99%
Epoch 15/20, Loss: -19.2079, Accuracy: 99.95%
Epoch 16/20, Loss: -20.4244, Accuracy: 100.00%
Epoch 17/20, Loss: -21.6433, Accuracy: 99.98%
Epoch 18/20, Loss: -22.8472, Accuracy: 99.80%
Epoch 19/20, Loss: -24.0075, Accuracy: 99.74%
Epoch 20/20, Loss: -25.1958, Accuracy: 100.00%
Final Accuracy on kfold: 100.00%


MIRO

In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import models, transforms, datasets

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------
# Configuration
# -------------
SOURCE_DOMAINS = ['photo', 'cartoon', 'sketch']
TARGET_DOMAIN = 'art_painting'
DATA_ROOT = './kfold'   # Directory that contains one folder per domain
NUM_CLASSES = 7         # Number of classes (assumed same across domains)
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 20
LAMBDA_MIRO = 0.1       # Weight for MIRO regularization

# -------------
# Transforms
# -------------
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# -------------------------------
# Define Model with Feature Extraction
# -------------------------------
class ResNet18WithFeatures(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # Remove last fc layer to obtain features
        self.features = nn.Sequential(*list(backbone.children())[:-1])  # (B, 512, 1, 1)
        self.classifier = nn.Linear(backbone.fc.in_features, num_classes)
        
    def forward(self, x):
        f = self.features(x)       # (B,512,1,1)
        f = torch.flatten(f, 1)    # (B,512)
        logits = self.classifier(f)
        return logits, f

# -------------------------------
# Define MIRO Regularization Module
# -------------------------------
class MIRORegularizer(nn.Module):
    def __init__(self, num_classes, feature_dim):
        """
        For each class, maintain learnable parameters for the "oracle" embedding:
         - mu: mean vector (initialized to zeros)
         - logvar: log diagonal covariance (initialized to zeros)
        """
        super().__init__()
        self.mu = nn.Parameter(torch.zeros(num_classes, feature_dim))
        self.logvar = nn.Parameter(torch.zeros(num_classes, feature_dim))
        
    def forward(self, features, labels):
        # Get oracle parameters for each sample’s class
        mu_target = self.mu[labels]              # (B, feature_dim)
        logvar_target = self.logvar[labels]        # (B, feature_dim)
        var_target = torch.exp(logvar_target)      # (B, feature_dim)
        d = features.size(1)
        # KL divergence between N(features, I) and N(mu_target, diag(var_target))
        kl_per_sample = 0.5 * (
            - torch.sum(logvar_target, dim=1) - d +
            torch.sum(1.0 / var_target, dim=1) +
            torch.sum((mu_target - features) ** 2 / var_target, dim=1)
        )
        return kl_per_sample.mean()

# -------------------------------
# Prepare Datasets and DataLoaders
# -------------------------------

# Create source datasets: for each domain in SOURCE_DOMAINS, load using ImageFolder.
source_datasets = []
for domain in SOURCE_DOMAINS:
    domain_path = os.path.join(DATA_ROOT, domain)
    if os.path.isdir(domain_path):
        ds = datasets.ImageFolder(domain_path, transform=transform_train)
        source_datasets.append(ds)
    else:
        print(f"Warning: Source domain folder {domain_path} not found.")
        
if len(source_datasets) == 0:
    raise ValueError("No valid source domains found.")

# Concatenate source datasets
source_dataset = ConcatDataset(source_datasets)

# Split the source dataset into training and validation subsets (e.g., 80/20 split)
n_total = len(source_dataset)
n_train = int(0.8 * n_total)
n_val = n_total - n_train
source_train_dataset, source_val_dataset = random_split(source_dataset, [n_train, n_val])

source_train_loader = DataLoader(source_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
source_val_loader = DataLoader(source_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Prepare target domain dataset (for final evaluation)
target_path = os.path.join(DATA_ROOT, TARGET_DOMAIN)
if not os.path.isdir(target_path):
    raise ValueError(f"Target domain folder {target_path} not found.")
target_dataset = datasets.ImageFolder(target_path, transform=transform_train)
target_loader = DataLoader(target_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# -------------------------------
# Initialize Model, MIRO Regularizer, Optimizer, and Loss Function
# -------------------------------
model = ResNet18WithFeatures(num_classes=NUM_CLASSES).to(DEVICE)
miro_reg = MIRORegularizer(num_classes=NUM_CLASSES, feature_dim=512).to(DEVICE)

# Use Adam optimizer to update both model and MIRO parameters
optimizer = optim.Adam(list(model.parameters()) + list(miro_reg.parameters()), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# -------------------------------
# Evaluation Function
# -------------------------------
def evaluate(loader, model, device):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits, _ = model(inputs)
            preds = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    model.train()
    return 100.0 * correct / total

# -------------------------------
# Training Loop
# -------------------------------
print("Starting training on source domains...")
for epoch in range(EPOCHS):
    running_loss = 0.0
    for inputs, labels in source_train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        
        # Forward pass
        logits, features = model(inputs)
        loss_cls = criterion(logits, labels)
        loss_miro = miro_reg(features, labels)
        loss = loss_cls + LAMBDA_MIRO * loss_miro
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(source_train_loader)
    val_acc = evaluate(source_val_loader, model, DEVICE)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss:.4f}, Source Val Acc: {val_acc:.2f}%")

# -------------------------------
# Final Evaluation on Target Domain (Art)
# -------------------------------
target_acc = evaluate(target_loader, model, DEVICE)
print(f"Final Accuracy on Target Domain ({TARGET_DOMAIN}): {target_acc:.2f}%")


Starting training on source domains...
Epoch 1/20, Loss: 8.7090, Source Val Acc: 20.64%
Epoch 2/20, Loss: 3.3626, Source Val Acc: 21.84%
Epoch 3/20, Loss: 1.6906, Source Val Acc: 25.61%
Epoch 4/20, Loss: 0.3721, Source Val Acc: 26.24%
Epoch 5/20, Loss: -0.9374, Source Val Acc: 26.94%
Epoch 6/20, Loss: -2.0273, Source Val Acc: 48.14%
Epoch 7/20, Loss: -3.4006, Source Val Acc: 68.28%
Epoch 8/20, Loss: -4.4663, Source Val Acc: 77.09%
Epoch 9/20, Loss: -5.7747, Source Val Acc: 83.13%
Epoch 10/20, Loss: -6.9282, Source Val Acc: 86.85%
Epoch 11/20, Loss: -7.9784, Source Val Acc: 86.97%
Epoch 12/20, Loss: -9.0022, Source Val Acc: 88.74%
Epoch 13/20, Loss: -9.9662, Source Val Acc: 90.12%
Epoch 14/20, Loss: -10.9204, Source Val Acc: 90.62%
Epoch 15/20, Loss: -11.8051, Source Val Acc: 90.12%
Epoch 16/20, Loss: -12.6694, Source Val Acc: 90.81%
Epoch 17/20, Loss: -13.5056, Source Val Acc: 91.57%
Epoch 18/20, Loss: -14.3375, Source Val Acc: 91.19%
Epoch 19/20, Loss: -15.1475, Source Val Acc: 91.32%

with ResNet50

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import models, transforms, datasets

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------
# Configuration
# -------------
SOURCE_DOMAINS = ['photo', 'cartoon', 'sketch']
TARGET_DOMAIN = 'art_painting'
DATA_ROOT = './kfold'   # Directory that contains one folder per domain
NUM_CLASSES = 7         # Number of classes (assumed same across domains)
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 20
LAMBDA_MIRO = 0.1       # Weight for MIRO regularization

# -------------
# Transforms
# -------------
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# -------------------------------
# Define Model with Feature Extraction
# -------------------------------
class ResNet50WithFeatures(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Remove last fc layer to obtain features
        self.features = nn.Sequential(*list(backbone.children())[:-1])  # (B, 512, 1, 1)
        self.classifier = nn.Linear(backbone.fc.in_features, num_classes)
        
    def forward(self, x):
        f = self.features(x)       # (B,512,1,1)
        f = torch.flatten(f, 1)    # (B,512)
        logits = self.classifier(f)
        return logits, f

# -------------------------------
# Define MIRO Regularization Module
# -------------------------------
class MIRORegularizer(nn.Module):
    def __init__(self, num_classes, feature_dim):
        """
        For each class, maintain learnable parameters for the "oracle" embedding:
         - mu: mean vector (initialized to zeros)
         - logvar: log diagonal covariance (initialized to zeros)
        """
        super().__init__()
        self.mu = nn.Parameter(torch.zeros(num_classes, feature_dim))
        self.logvar = nn.Parameter(torch.zeros(num_classes, feature_dim))
        
    def forward(self, features, labels):
        # Get oracle parameters for each sample’s class
        mu_target = self.mu[labels]              # (B, feature_dim)
        logvar_target = self.logvar[labels]        # (B, feature_dim)
        var_target = torch.exp(logvar_target)      # (B, feature_dim)
        d = features.size(1)
        # KL divergence between N(features, I) and N(mu_target, diag(var_target))
        kl_per_sample = 0.5 * (
            - torch.sum(logvar_target, dim=1) - d +
            torch.sum(1.0 / var_target, dim=1) +
            torch.sum((mu_target - features) ** 2 / var_target, dim=1)
        )
        return kl_per_sample.mean()

# -------------------------------
# Prepare Datasets and DataLoaders
# -------------------------------

# Create source datasets: for each domain in SOURCE_DOMAINS, load using ImageFolder.
source_datasets = []
for domain in SOURCE_DOMAINS:
    domain_path = os.path.join(DATA_ROOT, domain)
    if os.path.isdir(domain_path):
        ds = datasets.ImageFolder(domain_path, transform=transform_train)
        source_datasets.append(ds)
    else:
        print(f"Warning: Source domain folder {domain_path} not found.")
        
if len(source_datasets) == 0:
    raise ValueError("No valid source domains found.")

# Concatenate source datasets
source_dataset = ConcatDataset(source_datasets)

# Split the source dataset into training and validation subsets (e.g., 80/20 split)
n_total = len(source_dataset)
n_train = int(0.8 * n_total)
n_val = n_total - n_train
source_train_dataset, source_val_dataset = random_split(source_dataset, [n_train, n_val])

source_train_loader = DataLoader(source_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
source_val_loader = DataLoader(source_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Prepare target domain dataset (for final evaluation)
target_path = os.path.join(DATA_ROOT, TARGET_DOMAIN)
if not os.path.isdir(target_path):
    raise ValueError(f"Target domain folder {target_path} not found.")
target_dataset = datasets.ImageFolder(target_path, transform=transform_train)
target_loader = DataLoader(target_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# -------------------------------
# Initialize Model, MIRO Regularizer, Optimizer, and Loss Function
# -------------------------------
model = ResNet50WithFeatures(num_classes=NUM_CLASSES).to(DEVICE)
miro_reg = MIRORegularizer(num_classes=NUM_CLASSES, feature_dim=2048).to(DEVICE)

# Use Adam optimizer to update both model and MIRO parameters
optimizer = optim.Adam(list(model.parameters()) + list(miro_reg.parameters()), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# -------------------------------
# Evaluation Function
# -------------------------------
def evaluate(loader, model, device):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits, _ = model(inputs)
            preds = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    model.train()
    return 100.0 * correct / total

# -------------------------------
# Training Loop
# -------------------------------
print("Starting training on source domains...")
for epoch in range(EPOCHS):
    running_loss = 0.0
    for inputs, labels in source_train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        
        # Forward pass
        logits, features = model(inputs)
        loss_cls = criterion(logits, labels)
        loss_miro = miro_reg(features, labels)
        loss = loss_cls + LAMBDA_MIRO * loss_miro
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(source_train_loader)
    val_acc = evaluate(source_val_loader, model, DEVICE)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss:.4f}, Source Val Acc: {val_acc:.2f}%")

# -------------------------------
# Final Evaluation on Target Domain (Art)
# -------------------------------
target_acc = evaluate(target_loader, model, DEVICE)
print(f"Final Accuracy on Target Domain ({TARGET_DOMAIN}): {target_acc:.2f}%")


Starting training on source domains...
Epoch 1/20, Loss: 2.5243, Source Val Acc: 21.40%
Epoch 2/20, Loss: -3.4132, Source Val Acc: 23.16%


KeyboardInterrupt: 